# Dịch máy TED PT→EN — bản thực hành cơ bản

Vanilla RNN encoder/decoder, context cố định, greedy. Primary bleu ↑. Mặc định chế độ khôi phục đúng corpus neulab vì URL TFDS gốc hỏng; giữ nhánh tfds/local_jsonl.

Mô hình nhiệm vụ, embedding và tokenizer đều học từ train. Người dùng đã cho phép pretrained **riêng trong metric** COMET/BERTScore/ERRANT. Không dùng evaluator để tạo dự đoán hay cập nhật mô hình. Chạy độc lập bằng Restart Kernel → Run All; không cần notebook khác/fewshot/workbook. Chưa huấn luyện đầy đủ và không kèm điểm benchmark.

**Bản basic cập nhật:** Tokenizer khoảng trắng + vocabulary train-only thay SentencePiece/BPE; giữ RNN và đầy đủ mask/loss/checkpoint. Không phải bản tối ưu; không cam kết chênh lệch điểm.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Chuẩn bị thư viện

Dùng PyTorch của kernel. Python3.11/3.12 phù hợp khi cần TFDS/COMET. Không cài pretrained cho mô hình nhiệm vụ.


In [ ]:
import importlib.util
import subprocess
import sys
REQUIRED_PACKAGES = [('sacrebleu', 'sacrebleu==2.6.0'), ('datasets', 'datasets==4.8.5')]
missing = [package for module,package in REQUIRED_PACKAGES if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install",*missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Cần kernel có PyTorch CPU/CUDA phù hợp. Cài PyTorch rồi Restart Kernel → Run All.")
from pathlib import Path
from collections import Counter,defaultdict
from types import SimpleNamespace
import gc
import hashlib
import io
import json
import math
import random
import re
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence
from tqdm.auto import tqdm


## 2. Cấu hình cố định

Một GPU32GB là đích thiết kế. Không AMP, auto batch/VRAM hoặc OOM retry. Ngoại lệ pretrained chỉ cho evaluator đã được người dùng xác nhận. Hằng số sửa thủ công trước Run All.


In [ ]:
SEED = 42
VARIANT = "basic"
ARTIFACT_DIR = Path("artifacts/mt_ted_pt_en") / VARIANT
CACHE_DIR = ARTIFACT_DIR / "data_cache"
METRIC_CACHE_DIR = ARTIFACT_DIR / "metric_cache"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
TASK_PRETRAINING_POLICY = "scratch"
TASK_PRETRAINED_COMPONENTS = []
ALLOW_PRETRAINED_EVALUATORS = True  # Ngoại lệ người dùng cho phép chỉ dùng để chấm metric.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Thiết bị: {DEVICE}; biến thể: {VARIANT}; PyTorch {torch.__version__}; Python {sys.version.split()[0]}")


SOURCE = {'row': 1, 'task': 'Machine Translation PT→EN', 'dataset': 'TED HRLR Translate pt_to_en', 'tfds_builder': 'ted_hrlr_translate/pt_to_en:1.0.0', 'url': 'https://www.tensorflow.org/datasets/catalog/ted_hrlr_translate', 'recovery_dataset': 'neulab/ted_hrlr', 'recovery_revision': 'cf21722111c95aa73c30e13df430032db463736d'}
DATA_MODE = "hf_parquet_recovery"
LOCAL_DATA_DIR = Path("data/mt_ted_pt_en")
SOURCE["data_mode"] = DATA_MODE
EVALUATION_COMPONENTS = [{'role': 'evaluation_only', 'source': 'huggingface', 'model_id': 'Unbabel/wmt22-comet-da', 'revision': '2760a223ac957f30acfb18c8aa649b01cf1d75f2', 'training': 'frozen', 'purpose': 'COMET; chỉ final scoring'}, {'role': 'evaluation_only', 'source': 'huggingface', 'model_id': 'FacebookAI/xlm-roberta-large', 'revision': 'c23d21b0620b635a76227c604d44e43a9f0ee389', 'training': 'frozen', 'purpose': 'COMET encoder config/tokenizer; weights trong metric checkpoint'}]
VAL_FRACTION = 0.1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 32
VOCAB_SIZE = 8000
MAX_SOURCE_TOKENS = 128
MAX_TARGET_TOKENS = 128
MAX_DECODE_TOKENS = 128
EMBED_DIM = 128
HIDDEN_DIM = 256
DROPOUT = 0.2
BEAM_SIZE = 1
LENGTH_PENALTY = 0.0
LABEL_SMOOTHING = 0.0
EPOCHS = 45
PATIENCE = 8
LEARNING_RATE = 0.001
WARMUP_STEPS = 0
WEIGHT_DECAY = 0.0001
GRAD_CLIP = 1.0
PRIMARY_METRIC = "bleu"
RUN_COMET = True
COMET_BATCH_SIZE = 8
METRICS_OFFLINE = False


In [ ]:
def fingerprint(value):
    text=json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(",",":"))
    return hashlib.sha256(text.encode()).hexdigest()

def clean_text(text):
    return " ".join(unicodedata.normalize("NFC",text).split())

def read_jsonl(path):
    if not path.is_file():raise FileNotFoundError(f"Thiếu {path}; cần JSONL xuất từ đúng nguồn/schema đã ghi.")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def tfds_rows(builder,splits,supervised=False):
    # TensorFlow chỉ là reader CPU; PyTorch vẫn sử dụng DEVICE CUDA khi có.
    if importlib.util.find_spec("tensorflow") is None:
        if sys.version_info >= (3,13):
            raise RuntimeError("TFDS reader cần TensorFlow. Host Python này chưa có wheel phù hợp. "
                "Dùng kernel Python3.11/3.12 + tensorflow==2.18.1 tensorflow-datasets==4.9.10, "
                "hoặc chọn DATA_MODE='local_jsonl' với dữ liệu xuất đúng nguồn. Không có dữ liệu thay thế tự động.")
        subprocess.check_call([sys.executable,"-m","pip","install","tensorflow==2.18.1","tensorflow-datasets==4.9.10"])
    elif importlib.util.find_spec("tensorflow_datasets") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","tensorflow-datasets==4.9.10"])
    import tensorflow as tf
    try:tf.config.set_visible_devices([],"GPU")
    except RuntimeError as exc:
        raise RuntimeError("TensorFlow đã khởi tạo GPU. Restart Kernel rồi Run All để reader chạy CPU.") from exc
    import tensorflow_datasets as tfds
    try:
        data,info=tfds.load(builder,split=splits,as_supervised=supervised,with_info=True,
                            data_dir=str(CACHE_DIR),shuffle_files=False,try_gcs=True)
        print("TFDS:",builder,"; phiên bản thư viện",tfds.__version__,"; split counts",{k:v.num_examples for k,v in info.splits.items()})
        return {split:list(tfds.as_numpy(dataset)) for split,dataset in zip(splits,data)}
    except Exception as exc:
        raise RuntimeError(f"TFDS không đọc được {builder}. Kiểm tra cache/mạng/nguồn download; "
            "có thể dùng local_jsonl đã xuất từ đúng builder. Xem mục dữ liệu trong đề xuất, không đổi dataset âm thầm. "
            f"Chi tiết: {exc}") from exc

def decode_utf8(value):
    return value.decode("utf-8") if isinstance(value,(bytes,np.bytes_)) else str(value)

def to_device(batch):
    return {k:v.to(DEVICE) if torch.is_tensor(v) and k not in ("lengths","question_lengths") else v for k,v in batch.items()}

def make_loader(dataset,train=False):
    return DataLoader(dataset,batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE,
                      shuffle=train,num_workers=0,collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(SEED))


## 3. Dữ liệu và giao thức chia

Đọc đúng corpus/config, giữ test; loại source train trùng val/test trước fit. Với TFDS, reader chạy CPU để nhường CUDA cho PyTorch; không thay ngôn ngữ hay source âm thầm.


In [ ]:
def load_native():
    splits=["train","validation","test"]
    if DATA_MODE=="tfds":
        raw=tfds_rows(SOURCE["tfds_builder"],splits,supervised=True)
        return {split:[{"src":decode_utf8(pt),"refs":[decode_utf8(en)]} for pt,en in raw[split]] for split in splits}
    if DATA_MODE=="hf_parquet_recovery":
        # Cùng corpus của tác giả neulab, không đổi dataset; branch Parquet cố định.
        from datasets import load_dataset
        files={split:f"https://huggingface.co/datasets/neulab/ted_hrlr/resolve/{SOURCE['recovery_revision']}/pt_to_en/{split}/0000.parquet" for split in splits}
        try:raw=load_dataset("parquet",data_files=files,cache_dir=str(CACHE_DIR))
        except Exception as exc:raise RuntimeError(f"Không đọc được recovery TED cùng corpus. Dùng cache TFDS hoặc local_jsonl đúng PT–EN; không có dataset thay thế. {exc}") from exc
        output={}
        for split in splits:
            rows=[r["translation"] for r in raw[split]]
            # HF conversion có một dòng rỗng cuối mỗi split; builder TFDS bỏ hàng nếu một bên rỗng.
            output[split]=[{"src":r["pt"],"refs":[r["en"]]} for r in rows if r["pt"] and r["en"]]
        return output
    if DATA_MODE=="local_jsonl":
        return {split:[{"src":r["pt"],"refs":[r["en"]]} for r in read_jsonl(LOCAL_DATA_DIR/f"{split}.jsonl") if r["pt"] and r["en"]] for split in splits}
    raise ValueError("DATA_MODE phải là tfds, hf_parquet_recovery hoặc local_jsonl.")
native=load_native()


In [ ]:
# Giữ nguyên test, loại source train trùng val/test và source val trùng test.
for split,records in native.items():
    for record in records:
        record["src"]=clean_text(record["src"]);record["refs"]=[clean_text(ref) for ref in record["refs"]]
        if not record["src"] or not record["refs"] or any(not r for r in record["refs"]):raise ValueError(f"Mẫu rỗng ở{split}.")
test_sources={r["src"] for r in native["test"]}
removed_val=sum(r["src"] in test_sources for r in native["validation"])
native["validation"]=[r for r in native["validation"] if r["src"] not in test_sources]
unseen_sources=test_sources|{r["src"] for r in native["validation"]}
removed_train=sum(r["src"] in unseen_sources for r in native["train"])
native["train"]=[r for r in native["train"] if r["src"] not in unseen_sources]
def canonical_records(rows,split):
    output=[]
    for i,row in enumerate(sorted(rows,key=lambda r:(r["src"],tuple(r["refs"])))):
        output.append({**row,"id":f"{split}:{i}"})
    return output
train_records=canonical_records(native["train"],"train")
val_records=canonical_records(native["validation"],"validation")
test_records=canonical_records(native["test"],"test")
if any(not r for r in (train_records,val_records,test_records)):raise ValueError("Một split rỗng sau xử lý grouping.")
SPLIT_FINGERPRINTS={name:fingerprint([{"src":r["src"],"refs":r["refs"]} for r in records])
                    for name,records in (("train",train_records),("validation",val_records),("test",test_records))}
print("Loại source trùng: train",removed_train,"; validation",removed_val)
print("Số mẫu sau grouping:",len(train_records),len(val_records),len(test_records))
print("Fingerprint:",SPLIT_FINGERPRINTS)
print("Ví dụ train:",train_records[0])


## 4. Tokenizer train-only, Dataset và padding

Tách theo khoảng trắng và vocabulary train-only; giữ RNN đơn giản. Fit sau chia/grouping. Source/target có cap cố định và in số bị rút; gold evaluation nguyên vẹn. Train dùng mọi reference trong nhóm, eval một output/source.


In [ ]:

PAD_ID,UNK_ID,BOS_ID,EOS_ID=0,1,2,3

class BasicWordTokenizer:
    """Tách theo khoảng trắng; vocabulary chỉ từ train, không BPE hay pretrained."""
    def __init__(self,vocab):
        self.vocab=dict(vocab)
        self.words={index:word for word,index in self.vocab.items()}
    def encode(self,text,out_type=int):
        return [self.vocab.get(word,UNK_ID) for word in str(text).split()]
    def decode(self,ids):
        return " ".join(self.words.get(int(index),"<UNK>") for index in ids
                        if int(index) not in (PAD_ID,BOS_ID,EOS_ID))
    def get_piece_size(self):
        return len(self.vocab)

counts=Counter(word for record in train_records
               for text in [record["src"],*record["refs"]] for word in text.split())
words=sorted(counts,key=lambda word:(-counts[word],word))[:VOCAB_SIZE-4]
vocab={"<PAD>":PAD_ID,"<UNK>":UNK_ID,"<BOS>":BOS_ID,"<EOS>":EOS_ID}
vocab.update({word:index+4 for index,word in enumerate(words)})
sp=BasicWordTokenizer(vocab)
TOKENIZER_STATE={"vocab":vocab,"type":"whitespace words, train-only vocabulary",
                 "pad_id":PAD_ID,"unk_id":UNK_ID,"bos_id":BOS_ID,"eos_id":EOS_ID}
PREPROCESS_CONFIG={"max_source_tokens":MAX_SOURCE_TOKENS,"max_target_tokens":MAX_TARGET_TOKENS,
    "max_decode_tokens":MAX_DECODE_TOKENS,"normalization":"loader protocol unchanged; whitespace tokenization",
    "decoding":{"beam_size":BEAM_SIZE,"length_penalty":LENGTH_PENALTY}}

def restore_tokenizer(state):
    global sp
    sp=BasicWordTokenizer(state["vocab"])

class PairDataset(Dataset):
    def __init__(self,records,training=False):
        self.records=records
        self.index=[(i,j) for i,r in enumerate(records) for j in range(len(r["refs"]))] if training else [(i,0) for i in range(len(records))]
    def __len__(self):return len(self.index)
    def __getitem__(self,index):
        i,j=self.index[index];record=self.records[i]
        source=sp.encode(record["src"],out_type=int)[:MAX_SOURCE_TOKENS]
        if not source:source=[UNK_ID]
        target=[BOS_ID]+sp.encode(record["refs"][j],out_type=int)[:MAX_TARGET_TOKENS-2]+[EOS_ID]
        return torch.tensor(source),torch.tensor(target),record

def collate_batch(items):
    lengths=torch.tensor([len(src) for src,_,_ in items])
    source=torch.full((len(items),int(lengths.max())),PAD_ID,dtype=torch.long)
    target=torch.full((len(items),max(len(tgt) for _,tgt,_ in items)),PAD_ID,dtype=torch.long)
    for i,(src,tgt,_) in enumerate(items):source[i,:len(src)]=src;target[i,:len(tgt)]=tgt
    return {"source":source,"target":target,"lengths":lengths,
            "ids":[r["id"] for _,_,r in items],"records":[r for _,_,r in items]}

train_loader=make_loader(PairDataset(train_records,True),True)
val_loader=make_loader(PairDataset(val_records))
test_loader=make_loader(PairDataset(test_records))
print("Tokenizer vocab:",sp.get_piece_size(),"; fit chỉ train.")
for name,records in (("train",train_records),("validation",val_records),("test",test_records)):
    source_cut=sum(len(sp.encode(r["src"]))>MAX_SOURCE_TOKENS for r in records)
    target_cut=sum(len(sp.encode(ref))>MAX_TARGET_TOKENS-2 for r in records for ref in r["refs"])
    print(f"{name}: source quá cap={source_cut}; target quá cap train={target_cut}; gold metric giữ đầy đủ.")


## 5. Kiến trúc

Vanilla RNN encoder/decoder, context cố định, greedy. Không có text model/embedding/tokenizer pretrained trong forward.


In [ ]:
class TextGenerator(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.embedding=nn.Embedding(cfg["vocab_size"],cfg["embed_dim"],padding_idx=PAD_ID)
        self.encoder=nn.RNN(cfg["embed_dim"],cfg["hidden_dim"],batch_first=True)
        self.decoder=nn.RNNCell(cfg["embed_dim"],cfg["hidden_dim"])
        self.dropout=nn.Dropout(cfg["dropout"])
        self.output=nn.Linear(cfg["hidden_dim"],cfg["vocab_size"])
    def encode(self,source,lengths):
        packed=pack_padded_sequence(self.dropout(self.embedding(source)),lengths,batch_first=True,enforce_sorted=False)
        _,hidden=self.encoder(packed)
        return hidden[-1]
    def forward(self,source,lengths,target):
        hidden=self.encode(source,lengths);output=[]
        for t in range(target.size(1)):
            hidden=self.decoder(self.embedding(target[:,t]),hidden)
            output.append(self.output(self.dropout(hidden)).log_softmax(-1))
        return torch.stack(output,1)
    @torch.no_grad()
    def generate(self,source,lengths,max_tokens,beam_size=1,length_penalty=0.):
        hidden=self.encode(source,lengths)
        token=torch.full((len(source),),BOS_ID,device=source.device,dtype=torch.long)
        done=torch.zeros(len(source),device=source.device,dtype=torch.bool);output=[]
        for _ in range(max_tokens):
            hidden=self.decoder(self.embedding(token),hidden)
            logits=self.output(hidden);logits[:,[PAD_ID,BOS_ID]]=-torch.inf
            token=logits.argmax(-1);token=torch.where(done,torch.full_like(token,EOS_ID),token)
            output.append(token);done|=token==EOS_ID
            if bool(done.all()):break
        return torch.stack(output,1)
def build_model(cfg):return TextGenerator(cfg)


In [ ]:
MODEL_CONFIG={"vocab_size":sp.get_piece_size(),"embed_dim":EMBED_DIM,"hidden_dim":HIDDEN_DIM,"dropout":DROPOUT}


## 6. Metric, loss và giải mã

Teacher forcing shift BOS/EOS, bỏ PAD. Causal/key-padding masks cho Transformer; packed sequence/attention PAD mask cho RNN. Beam/greedy chỉ đọc source. Primary được chọn theo metric đầu hàng; không chọn checkpoint trên metric phụ hay test.


In [ ]:
from sacrebleu.metrics import BLEU,CHRF
bleu_metric=BLEU(tokenize="13a",smooth_method="exp",effective_order=False)
chrf_metric=CHRF(word_order=2)
def score_corpus(records,predictions):
    refs=[[r["refs"][0] for r in records]]
    b=bleu_metric.corpus_score(predictions,refs)
    c=chrf_metric.corpus_score(predictions,refs)
    return {"bleu":float(b.score),"chrfpp":float(c.score),
            "bleu_signature":str(bleu_metric.get_signature()),"chrf_signature":str(chrf_metric.get_signature())}


In [ ]:
def decode_tokens(ids):
    output=[]
    for index in ids:
        index=int(index)
        if index==EOS_ID:break
        if index not in (PAD_ID,BOS_ID):output.append(index)
    return clean_text(sp.decode(output))
def compute_loss(model,batch):
    logp=model(batch["source"],batch["lengths"],batch["target"][:,:-1])
    target=batch["target"][:,1:]
    losses=-logp.gather(-1,target.unsqueeze(-1)).squeeze(-1)
    if LABEL_SMOOTHING:losses=(1-LABEL_SMOOTHING)*losses+LABEL_SMOOTHING*(-logp.mean(-1))
    mask=target!=PAD_ID
    return losses[mask].mean()

@torch.no_grad()
def evaluate(model,loader,return_predictions=False):
    model.eval();records=[];guesses=[]
    for cpu_batch in loader:
        batch=to_device(cpu_batch)
        outputs=model.generate(batch["source"],batch["lengths"],MAX_DECODE_TOKENS,BEAM_SIZE,LENGTH_PENALTY).cpu().tolist()
        records.extend(batch["records"]);guesses.extend(decode_tokens(ids) for ids in outputs)
    if not records:raise ValueError("Không có mẫu đánh giá.")
    metrics=score_corpus(records,guesses)
    rows=[{"id":r["id"],"source":r["src"],"prediction":prediction,"references":r["refs"]} for r,prediction in zip(records,guesses)]
    return (metrics,rows) if return_predictions else metrics


In [ ]:
if RUN_COMET and importlib.util.find_spec("comet") is None:
    raise RuntimeError("RUN_COMET=True: cần cài unbabel-comet==2.2.7 vào kernel Python3.11/3.12 trước train. Có thể chủ động tắt RUN_COMET để chỉ đo BLEU/chrF++ và ghi COMET chưa đo.")


## 7. Huấn luyện và checkpoint

AdamW, warmup/inverse-sqrt chỉ khi WARMUP_STEPS>0; clipping, early stopping primary validation tăng. Lưu task weights, optimizer/scheduler, model/tokenizer/config/provenance/fingerprints; evaluator weights chỉ trong cache metric riêng.


In [ ]:
model=build_model(MODEL_CONFIG).to(DEVICE)
print(f"Tham số nhiệm vụ học từ đầu: {sum(p.numel() for p in model.parameters()):,}")
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,betas=(0.9,0.98),eps=1e-9,weight_decay=WEIGHT_DECAY)
total_steps=max(1,EPOCHS*len(train_loader))
def lr_multiplier(step):
    step+=1
    return min(step/max(1,WARMUP_STEPS),math.sqrt(max(1,WARMUP_STEPS)/max(step,WARMUP_STEPS))) if WARMUP_STEPS else 1.0
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_multiplier)
best_score=-math.inf;stale_epochs=0
for epoch in range(1,EPOCHS+1):
    model.train();total=0.;seen=0
    for cpu_batch in tqdm(train_loader,desc="Huấn luyện",leave=False):
        batch=to_device(cpu_batch)
        optimizer.zero_grad(set_to_none=True)
        loss=compute_loss(model,batch)
        if not torch.isfinite(loss):raise FloatingPointError("Loss không hữu hạn; kiểm tra mask/nhãn và cấu hình cố định.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
        optimizer.step();scheduler.step()
        n=len(batch["ids"]);total+=loss.item()*n;seen+=n
    if not seen:raise ValueError("Train rỗng.")
    loss = total/seen
    metrics=evaluate(model,val_loader)
    score=float(metrics[PRIMARY_METRIC])
    if not math.isfinite(score):raise ValueError("Primary validation metric không hữu hạn.")
    print(f"Epoch{epoch:02d}: loss={loss:.4f}; validation={metrics}")
    if score>best_score:
        best_score=score;stale_epochs=0
        torch.save({
            "model_state": model.state_dict(),
            "tokenizer": TOKENIZER_STATE,
            "epoch": epoch,
        }, CHECKPOINT_PATH)
        print("Đã lưu checkpoint tốt nhất theo primary metric validation tăng.")
    else:
        stale_epochs+=1
        if stale_epochs>=PATIENCE:
            print(f"Dừng sớm sau {PATIENCE} epoch không cải thiện.");break


## 8. Tải lại và đánh giá held-out

Khởi tạo lại model và tokenizer từ checkpoint rồi chấm test/held-out một lần. Không hứa điểm hoặc chênh lệch chưa chạy.


In [ ]:
checkpoint=torch.load(CHECKPOINT_PATH,map_location=DEVICE,weights_only=True)
restore_tokenizer(checkpoint["tokenizer"])
# Cấu hình kiến trúc ở notebook; kích thước vocab được lấy sau khi restore.
inference_config = {"vocab_size":sp.get_piece_size(),"embed_dim":EMBED_DIM,"hidden_dim":HIDDEN_DIM,"dropout":DROPOUT}
best_model=build_model(inference_config).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"])
best_model.eval()
validation_metrics=evaluate(best_model,val_loader)
test_metrics,predictions=evaluate(best_model,test_loader,return_predictions=True)
results={"source":SOURCE,"epoch":checkpoint["epoch"],"validation":validation_metrics,
         "held_out":test_metrics,"split_fingerprints":SPLIT_FINGERPRINTS}
(ARTIFACT_DIR/"metrics.json").write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding="utf-8")
(ARTIFACT_DIR/"predictions.json").write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")
print("Đã nạp checkpoint epoch",checkpoint["epoch"])
print("Validation:",validation_metrics)
print("Held-out:",test_metrics)
print("Ví dụ:",predictions[0] if predictions else "Không có mẫu")


In [ ]:
def add_comet_metric(rows):
    if not RUN_COMET:return {"comet":None,"comet_status":"Người chạy tắt RUN_COMET; chưa đo, không phải0."}
    if not ALLOW_PRETRAINED_EVALUATORS:raise RuntimeError("Chưa cho phép pretrained evaluator.")
    if importlib.util.find_spec("comet") is None:
        raise RuntimeError("Cần unbabel-comet==2.2.7 trong kernel Python3.11/3.12 phù hợp. "
                           "Cài trước Run All; nếu chủ động không chấm, đặt RUN_COMET=False và kết quả sẽ ghi chưa đo.")
    from huggingface_hub import snapshot_download
    from comet.models.regression.regression_metric import RegressionMetric
    model_path=snapshot_download("Unbabel/wmt22-comet-da",revision="2760a223ac957f30acfb18c8aa649b01cf1d75f2",
        cache_dir=str(METRIC_CACHE_DIR),local_files_only=METRICS_OFFLINE,
        allow_patterns=["checkpoints/model.ckpt","hparams.yaml","README.md"])
    encoder_path=snapshot_download("FacebookAI/xlm-roberta-large",revision="c23d21b0620b635a76227c604d44e43a9f0ee389",
        cache_dir=str(METRIC_CACHE_DIR),local_files_only=METRICS_OFFLINE,
        allow_patterns=["config.json","tokenizer.json","tokenizer_config.json","sentencepiece.bpe.model","special_tokens_map.json"])
    evaluator=RegressionMetric.load_from_checkpoint(str(Path(model_path)/"checkpoints/model.ckpt"),map_location="cpu",
        load_pretrained_weights=False,pretrained_model=str(encoder_path),local_files_only=True,strict=False)
    evaluator.eval()
    for p in evaluator.parameters():p.requires_grad=False
    data=[{"src":r["source"],"mt":r["prediction"],"ref":r["references"][0]} for r in rows]
    output=evaluator.predict(data,batch_size=COMET_BATCH_SIZE,gpus=1 if DEVICE.type=="cuda" else 0,num_workers=0)
    score=float(output.system_score)
    del evaluator;gc.collect()
    return {"comet":score,"comet_model":"Unbabel/wmt22-comet-da@2760a223ac957f30acfb18c8aa649b01cf1d75f2",
            "comet_status":"Đã đo; raw model score, không nhân100; không phải BLEU"}

# Sau primary held-out evaluation, chỉ chấm evaluator trên output đã cố định.
extra=add_comet_metric(predictions)
results["held_out"].update(extra)
(ARTIFACT_DIR/"metrics.json").write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding="utf-8")
print("COMET held-out:",extra)


## 9. Suy luận văn bản mới

Không có gold reference hoặc evaluator nào tham gia tạo dự đoán. Input dài có thể bị rút theo MAX_SOURCE_TOKENS, cần lưu ý khi ứng dụng.


In [ ]:
@torch.no_grad()
def rewrite_text(text):
    text=clean_text(text)
    if not text:raise ValueError("Cần văn bản không rỗng.")
    ids=sp.encode(text,out_type=int)[:MAX_SOURCE_TOKENS]
    source=torch.tensor([ids or [UNK_ID]],device=DEVICE)
    best_model.eval()
    output=best_model.generate(source,torch.tensor([source.size(1)]),MAX_DECODE_TOKENS,BEAM_SIZE,LENGTH_PENALTY)
    return decode_tokens(output[0].tolist())

print(rewrite_text(test_records[0]["src"]))
